# gpt-oss-20b Unsloth 全流程

这个 notebook 说明 gpt-oss-20b 使用 Unsloth 的训练、推理、保存和导出路线。

gpt-oss 场景中 Unsloth 的价值是封装 Harmony、精度、低显存训练和导出细节，降低手动踩坑概率。

## 1. 重点注意

- 确认当前 Unsloth 版本支持 gpt-oss。
- 使用正确 Harmony/chat template。
- 优先 bf16，谨慎 fp16。
- reasoning effort 要在推理验证中明确设置。
- MXFP4 和 bitsandbytes 4bit 不是同一个概念。

In [ ]:
# 安装支持 gpt-oss 的 unsloth 后可执行。
# from unsloth import FastLanguageModel
#
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="openai/gpt-oss-20b",
#     max_seq_length=2048,
#     dtype=None,
#     load_in_4bit=True,
# )

## 2. LoRA/QLoRA

gpt-oss-20b 不建议新人直接全量微调。先用 LoRA 或 QLoRA 跑通数据、模板和 loss。

In [ ]:
# model = FastLanguageModel.get_peft_model(
#     model,
#     r=16,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     lora_alpha=32,
#     lora_dropout=0.05,
#     bias="none",
#     use_gradient_checkpointing="unsloth",
#     random_state=42,
# )

## 3. SFT 与保存

训练数据仍然可以从 `data/sample_sft.jsonl` 开始，但正式训练前必须转换或确认 Harmony 格式正确。

In [ ]:
# from datasets import load_dataset
# from transformers import TrainingArguments
# from trl import SFTTrainer
#
# dataset = load_dataset("json", data_files="../../data/sample_sft.jsonl", split="train")
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=dataset,
#     args=TrainingArguments(
#         output_dir="../../outputs/gpt_oss_20b/unsloth_sft",
#         per_device_train_batch_size=1,
#         gradient_accumulation_steps=16,
#         learning_rate=1e-4,
#         num_train_epochs=1,
#         logging_steps=1,
#         report_to="none",
#     ),
# )
# trainer.train()
# model.save_pretrained("../../outputs/gpt_oss_20b/unsloth_adapter")
# tokenizer.save_pretrained("../../outputs/gpt_oss_20b/unsloth_adapter")

## 4. 推理验证

保存前后都要验证输出。gpt-oss 验证时要检查 reasoning effort 和最终回答格式。

In [ ]:
# FastLanguageModel.for_inference(model)
# messages = [{"role": "user", "content": "解释 MoE 的 active parameters。"}]
# prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))